In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-6")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(
    request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)

    return handler(request)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern.",
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Did you water the office plant today?")]}
)

print(response["messages"][-1].content)

I don’t have visibility into the office plant, so I didn’t actually water it today. If you’d like, I can help set up a quick system so it doesn’t slip again.

Options:
- I can create a simple watering reminder (e.g., every 7–10 days) and a basic log to track when it’s watered.
- I can give you a quick care checklist for common office plants (soil moisture check, how much to water, drainage tips).
- I can draft a short template you can share with others who water it.

Would you like me to set up a reminder and a watering log, or give you a quick care guide for your plant? Also, what plant is it? It helps tailor the watering frequency.


In [5]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [6]:
from langchain.messages import AIMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Did you water the office plant today?"),
            AIMessage(content="Yes, I gave it a light watering this morning."),
            HumanMessage(content="Has it grown much this week?"),
            AIMessage(content="It's sprouted two new leaves since Monday."),
            HumanMessage(content="Are the leaves still turning yellow on the edges?"),
            AIMessage(content="A little, but it's looking healthier overall."),
            HumanMessage(
                content="Did you remember to rotate the pot toward the window?"
            ),
            AIMessage(
                content="I rotated it a quarter turn so it gets more even light."
            ),
            HumanMessage(content="How often should we be fertilizing this plant?"),
            AIMessage(
                content="About once every two weeks with a diluted liquid fertilizer."
            ),
            HumanMessage(content="When should we expect to have to replace the pot?"),
        ]
    }
)

print(response["messages"][-1].content)

It's hard to say for sure - I'd keep an eye on the roots. If you start to see them poking out of the drainage holes or circling the bottom of the pot, that's usually a good sign it's time to repot. Could be anywhere from a few months to a year or more depending on how fast it grows. I can check on that next time I water it if you'd like!


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

claude-sonnet-4-6
